# Notebook 04 — Embedding and Vector DB Tests

## Goal
This notebook tests local embedding generation with LM Studio and vector storage with Qdrant.

## Main tasks
- Reload cleaned documents
- Apply selected chunking strategy
- Connect to LM Studio embeddings endpoint
- Generate embeddings locally
- Create Qdrant collection in local mode
- Store chunks with metadata
- Run similarity search queries

In [1]:
# Standard library
from pathlib import Path
from typing import List, Dict, Any
import re
import json
from copy import deepcopy
import uuid

# Data handling
import pandas as pd

# OpenAI-compatible client for LM Studio
from openai import OpenAI

# LangChain loaders
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import TextLoader

# LangChain splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Imports loaded successfully.")

Imports loaded successfully.


In [2]:
# ============================================================
# STEP 1 — Define paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "chunked_docs"
QDRANT_DIR = PROJECT_ROOT / "vectorstore" / "qdrant_data"

QDRANT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("QDRANT_DIR:", QDRANT_DIR)

PROJECT_ROOT: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag
DATA_DIR: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\raw
OUTPUT_DIR: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs
QDRANT_DIR: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\vectorstore\qdrant_data


In [4]:
# ============================================================
# STEP 2 — LM Studio connection settings
# ============================================================

LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
LM_STUDIO_API_KEY = "lm-studio"

# IMPORTANT:
# Replace this with the exact embedding model name visible in LM Studio
EMBEDDING_MODEL_NAME = "darrowoflykos/google_embeddinggemma-300m-qat-q8_0-GGUF"

print("LM Studio base URL:", LM_STUDIO_BASE_URL)
print("Embedding model placeholder:", EMBEDDING_MODEL_NAME)

LM Studio base URL: http://localhost:1234/v1
Embedding model placeholder: darrowoflykos/google_embeddinggemma-300m-qat-q8_0-GGUF


In [5]:
# ============================================================
# STEP 3 — File type detection and loading
# ============================================================

def detect_file_type(file_path: Path) -> str:
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return "pdf"
    elif suffix == ".docx":
        return "docx"
    elif suffix == ".txt":
        return "txt"
    else:
        return "unsupported"


def load_single_document(file_path: Path):
    file_type = detect_file_type(file_path)

    if file_type == "pdf":
        loader = PyPDFLoader(str(file_path))
        docs = loader.load()

    elif file_type == "docx":
        loader = Docx2txtLoader(str(file_path))
        docs = loader.load()

    elif file_type == "txt":
        loader = TextLoader(str(file_path), encoding="utf-8")
        docs = loader.load()

    else:
        raise ValueError(f"Unsupported file type: {file_path.suffix}")

    return docs

In [6]:
# ============================================================
# STEP 4 — Cleaning helpers
# ============================================================

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\t", " ")
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()

    return text


def remove_light_page_noise(text: str) -> str:
    if not isinstance(text, str):
        return ""

    lines = text.split("\n")
    cleaned_lines = [line.strip() for line in lines]
    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

In [7]:
# ============================================================
# STEP 5 — Standard metadata schema
# ============================================================

def build_standard_metadata(
    original_metadata: Dict[str, Any],
    global_doc_index: int
) -> Dict[str, Any]:
    source_path = str(original_metadata.get("source", "")).strip()
    source_obj = Path(source_path) if source_path else None

    file_name = source_obj.name if source_obj else "unknown"
    file_extension = source_obj.suffix.lower() if source_obj else ""
    doc_type = file_extension.replace(".", "") if file_extension else "unknown"

    page_number = original_metadata.get("page", None)

    standardized = {
        "document_id": f"doc_{global_doc_index:05d}",
        "file_name": file_name,
        "source_path": source_path,
        "doc_type": doc_type,
        "page_number": page_number,
        "section_title": None,
        "chunk_id": None,
        "loader_name": (
            "PyPDFLoader" if doc_type == "pdf"
            else "Docx2txtLoader" if doc_type == "docx"
            else "TextLoader" if doc_type == "txt"
            else "unknown"
        ),
        "original_metadata": dict(original_metadata)
    }

    return standardized

In [8]:
# ============================================================
# STEP 6 — Reload and clean source documents
# ============================================================

all_files = sorted([p for p in DATA_DIR.iterdir() if p.is_file()])

raw_documents = []

for file_path in all_files:
    if detect_file_type(file_path) == "unsupported":
        continue

    docs = load_single_document(file_path)
    raw_documents.extend(docs)

cleaned_documents = []

for idx, doc in enumerate(raw_documents):
    doc.page_content = remove_light_page_noise(clean_text(doc.page_content))
    doc.metadata = build_standard_metadata(doc.metadata, idx)
    cleaned_documents.append(doc)

print("Total cleaned documents:", len(cleaned_documents))

Total cleaned documents: 114


In [9]:
# ============================================================
# STEP 7 — Chunk documents using selected strategy
# ============================================================

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SEPARATORS
)

chunked_documents = []

for doc in cleaned_documents:
    split_docs = splitter.split_documents([doc])

    for chunk_index, chunk_doc in enumerate(split_docs):
        chunk_doc.metadata = deepcopy(chunk_doc.metadata)
        chunk_doc.metadata["parent_document_id"] = doc.metadata.get("document_id")
        chunk_doc.metadata["chunk_id"] = f"{doc.metadata.get('document_id')}_chunk_{chunk_index:03d}"
        chunk_doc.metadata["chunk_size"] = CHUNK_SIZE
        chunk_doc.metadata["chunk_overlap"] = CHUNK_OVERLAP
        chunked_documents.append(chunk_doc)

print("Total chunked documents:", len(chunked_documents))

Total chunked documents: 613


In [10]:
# ============================================================
# STEP 8 — Preview chunk samples
# ============================================================

preview_rows = []

for doc in chunked_documents[:15]:
    preview_rows.append({
        "chunk_id": doc.metadata.get("chunk_id"),
        "file_name": doc.metadata.get("file_name"),
        "doc_type": doc.metadata.get("doc_type"),
        "page_number": doc.metadata.get("page_number"),
        "char_count": len(doc.page_content),
        "word_count": len(doc.page_content.split()),
        "text_preview": doc.page_content[:200].replace("\n", " ")
    })

preview_df = pd.DataFrame(preview_rows)
display(preview_df)

,chunk_id,file_name,doc_type,page_number,char_count,word_count,text_preview
0,doc_00000_chunk_000,audit.txt,txt,None,501,74,AUDIT TRAILS Audit trails maintain a record o...
1,doc_00000_chunk_001,audit.txt,txt,None,466,73,An audit trail is a series of records of compu...
2,doc_00000_chunk_002,audit.txt,txt,None,676,104,Audit trails may be used as either a support f...
3,doc_00000_chunk_003,audit.txt,txt,None,433,66,Individual Accountability Audit trails are a t...
4,doc_00000_chunk_004,audit.txt,txt,None,623,101,"For example, audit trails can be used in conce..."
5,doc_00000_chunk_005,audit.txt,txt,None,795,126,Audit trails work in concert with logical acce...
6,doc_00000_chunk_006,audit.txt,txt,None,327,55,personal data. Another example may be an engin...
7,doc_00000_chunk_007,audit.txt,txt,None,737,117,Reconstruction of Events Audit trails can also...
8,doc_00000_chunk_008,audit.txt,txt,None,351,59,"users, and the application. Knowledge of the c..."
9,doc_00000_chunk_009,audit.txt,txt,None,538,83,Intrusion Detection Intrusion detection refers...


In [11]:
# ============================================================
# STEP 9 — Create LM Studio OpenAI-compatible client
# ============================================================

client = OpenAI(
    base_url=LM_STUDIO_BASE_URL,
    api_key=LM_STUDIO_API_KEY
)

print("LM Studio client created successfully.")

LM Studio client created successfully.


In [12]:
# ============================================================
# STEP 10 — Test embedding generation on one chunk
# ============================================================

test_text = chunked_documents[0].page_content if chunked_documents else "Test embedding text."

response = client.embeddings.create(
    model=EMBEDDING_MODEL_NAME,
    input=test_text
)

test_embedding = response.data[0].embedding

print("Embedding generated successfully.")
print("Embedding dimension:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding generated successfully.
Embedding dimension: 768
First 10 values: [0.059559013694524765, -0.006359780672937632, -0.021130651235580444, 0.010661346837878227, 0.012527612037956715, -0.013049651868641376, -0.0023013560567051172, 0.019937073811888695, -0.015457071363925934, -0.04609014093875885]


In [13]:
# ============================================================
# STEP 11 — Embedding helper function
# ============================================================

def get_embedding(text: str, model_name: str) -> List[float]:
    response = client.embeddings.create(
        model=model_name,
        input=text
    )
    return response.data[0].embedding

In [14]:
# ============================================================
# STEP 12 — Generate embeddings for all chunks
# ============================================================

embedded_records = []

for i, doc in enumerate(chunked_documents):
    vector = get_embedding(doc.page_content, EMBEDDING_MODEL_NAME)

    embedded_records.append({
        "id": str(uuid.uuid4()),
        "vector": vector,
        "text": doc.page_content,
        "metadata": doc.metadata
    })

    if (i + 1) % 10 == 0 or (i + 1) == len(chunked_documents):
        print(f"Embedded {i + 1}/{len(chunked_documents)} chunks")

print("Total embedded records:", len(embedded_records))

Embedded 10/613 chunks
Embedded 20/613 chunks
Embedded 30/613 chunks
Embedded 40/613 chunks
Embedded 50/613 chunks
Embedded 60/613 chunks
Embedded 70/613 chunks
Embedded 80/613 chunks
Embedded 90/613 chunks
Embedded 100/613 chunks
Embedded 110/613 chunks
Embedded 120/613 chunks
Embedded 130/613 chunks
Embedded 140/613 chunks
Embedded 150/613 chunks
Embedded 160/613 chunks
Embedded 170/613 chunks
Embedded 180/613 chunks
Embedded 190/613 chunks
Embedded 200/613 chunks
Embedded 210/613 chunks
Embedded 220/613 chunks
Embedded 230/613 chunks
Embedded 240/613 chunks
Embedded 250/613 chunks
Embedded 260/613 chunks
Embedded 270/613 chunks
Embedded 280/613 chunks
Embedded 290/613 chunks
Embedded 300/613 chunks
Embedded 310/613 chunks
Embedded 320/613 chunks
Embedded 330/613 chunks
Embedded 340/613 chunks
Embedded 350/613 chunks
Embedded 360/613 chunks
Embedded 370/613 chunks
Embedded 380/613 chunks
Embedded 390/613 chunks
Embedded 400/613 chunks
Embedded 410/613 chunks
Embedded 420/613 chunks
E

In [15]:
# ============================================================
# STEP 13 — Embedding summary
# ============================================================

embedding_dim = len(embedded_records[0]["vector"]) if embedded_records else 0

embedding_summary_df = pd.DataFrame([{
    "num_records": len(embedded_records),
    "embedding_dimension": embedding_dim,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "model_name": EMBEDDING_MODEL_NAME
}])

display(embedding_summary_df)

,num_records,embedding_dimension,chunk_size,chunk_overlap,model_name
0,613,768,800,120,darrowoflykos/google_embeddinggemma-300m-qat-q...


In [16]:
# ============================================================
# STEP 14 — Create Qdrant local client
# ============================================================

qdrant_client = QdrantClient(path=str(QDRANT_DIR))
print("Qdrant local client created successfully.")

Qdrant local client created successfully.


In [17]:
# ============================================================
# STEP 15 — Create or recreate collection
# ============================================================

COLLECTION_NAME = "cybersecurity_rag_chunks"

qdrant_client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=embedding_dim,
        distance=Distance.COSINE
    )
)

print(f"Collection '{COLLECTION_NAME}' created successfully.")

Collection 'cybersecurity_rag_chunks' created successfully.


C:\Users\ABHISHEK\AppData\Local\Temp\ipykernel_69020\2927492809.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


In [18]:
# ============================================================
# STEP 16 — Convert records into Qdrant points
# ============================================================

points = []

for record in embedded_records:
    payload = {
        "text": record["text"],
        **record["metadata"]
    }

    points.append(
        PointStruct(
            id=record["id"],
            vector=record["vector"],
            payload=payload
        )
    )

print("Prepared points:", len(points))

Prepared points: 613


In [19]:
# ============================================================
# STEP 17 — Upload points to Qdrant
# ============================================================

qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print("Points uploaded successfully.")

Points uploaded successfully.


In [20]:
# ============================================================
# STEP 18 — Inspect collection info
# ============================================================

collection_info = qdrant_client.get_collection(COLLECTION_NAME)
print(collection_info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=613 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_

In [23]:
# ============================================================
# STEP 19 — Query helper
# ============================================================

def search_similar_chunks(
    query_text: str,
    top_k: int = 5
):
    query_vector = get_embedding(query_text, EMBEDDING_MODEL_NAME)

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k
    )

    # In newer Qdrant client versions, matches are usually in .points
    if hasattr(results, "points"):
        return results.points

    # Safe fallback in case the returned object is already list-like
    return results

In [24]:
# ============================================================
# STEP 20 — Run a sample search query
# ============================================================

sample_query = "What does the incident handling guide say about reporting and response procedures?"

results = search_similar_chunks(sample_query, top_k=5)

print(f"Retrieved {len(results)} results for query:")
print(sample_query)

Retrieved 5 results for query:
What does the incident handling guide say about reporting and response procedures?


In [25]:
# ============================================================
# STEP 21 — Display search results
# ============================================================

result_rows = []

for rank, result in enumerate(results, start=1):
    payload = result.payload

    result_rows.append({
        "rank": rank,
        "score": result.score,
        "file_name": payload.get("file_name"),
        "doc_type": payload.get("doc_type"),
        "page_number": payload.get("page_number"),
        "chunk_id": payload.get("chunk_id"),
        "parent_document_id": payload.get("parent_document_id"),
        "text_preview": payload.get("text", "")[:250].replace("\n", " ")
    })

results_df = pd.DataFrame(result_rows)
display(results_df)

,rank,score,file_name,doc_type,page_number,chunk_id,parent_document_id,text_preview
0,1,0.780320,nist.sp.800-61r2.pdf,pdf,28,doc_00062_chunk_002,doc_00062,"reporting incidents, among other items.  Deve..."
1,2,0.727059,nist.sp.800-61r2.pdf,pdf,5,doc_00039_chunk_000,doc_00039,COMPUTER SECURITY INCIDENT HANDLING GUIDE iv A...
2,3,0.721181,nist.sp.800-61r2.pdf,pdf,10,doc_00044_chunk_001,doc_00044,"To that end, this publication provides guideli..."
3,4,0.720061,nist.sp.800-61r2.pdf,pdf,42,doc_00076_chunk_005,doc_00076,management. This process should be repeated un...
4,5,0.712315,nist.sp.800-61r2.pdf,pdf,3,doc_00037_chunk_000,doc_00037,COMPUTER SECURITY INCIDENT HANDLING GUIDE ii ...


In [26]:
# ============================================================
# STEP 22 — Inspect full retrieved chunks
# ============================================================

for rank, result in enumerate(results, start=1):
    payload = result.payload

    print("=" * 120)
    print(f"RANK: {rank}")
    print("SCORE:", result.score)
    print("FILE:", payload.get("file_name"))
    print("DOC TYPE:", payload.get("doc_type"))
    print("PAGE:", payload.get("page_number"))
    print("CHUNK ID:", payload.get("chunk_id"))
    print("\nTEXT:\n")
    print(payload.get("text", "")[:1500])
    print("\n")

RANK: 1
SCORE: 0.7803203343047711
FILE: nist.sp.800-61r2.pdf
DOC TYPE: pdf
PAGE: 28
CHUNK ID: doc_00062_chunk_002

TEXT:

reporting incidents, among other items.
 Develop an incident response plan based on the incident response policy. The incident response
plan provides a roadmap for implementing an incident response program based on the organization’s
policy. The plan indicates both short- and long-term goals for the program, including metrics for
measuring the program. The incident response plan should also indicate how often incident handlers
should be trained and the requirements for incident handlers.
 Develop incident response procedures. The incident response procedures provide detailed steps for
responding to an incident. The procedures should cover all the phases of the incident response
process. The procedures should be based on the incident response policy and plan.


RANK: 2
SCORE: 0.7270588535378937
FILE: nist.sp.800-61r2.pdf
DOC TYPE: pdf
PAGE: 5
CHUNK ID: doc_00039_ch

In [27]:
# ============================================================
# STEP 23 — Evaluate multiple example queries
# ============================================================

test_queries = [
    "What are the main functions or categories in the cybersecurity framework?",
    "What steps are described for incident response or handling?",
    "What kind of system security planning information appears in the CUI SSP template?",
    "What information is present in the audit trail text?"
]

multi_query_rows = []

for query in test_queries:
    retrieved = search_similar_chunks(query, top_k=3)

    for rank, result in enumerate(retrieved, start=1):
        payload = result.payload

        multi_query_rows.append({
            "query": query,
            "rank": rank,
            "score": result.score,
            "file_name": payload.get("file_name"),
            "doc_type": payload.get("doc_type"),
            "page_number": payload.get("page_number"),
            "chunk_id": payload.get("chunk_id"),
            "text_preview": payload.get("text", "")[:180].replace("\n", " ")
        })

multi_query_df = pd.DataFrame(multi_query_rows)
display(multi_query_df)

,query,rank,score,file_name,doc_type,page_number,chunk_id,text_preview
0,What are the main functions or categories in t...,1,0.703991,NIST.CSWP.29.pdf,pdf,9.0,doc_00011_chunk_001,Fig. 2. CSF Functions The Functions should be ...
1,What are the main functions or categories in t...,2,0.703605,NIST.CSWP.29.pdf,pdf,8.0,doc_00010_chunk_001,"management, authentication, and access control..."
2,What are the main functions or categories in t...,3,0.684551,NIST.CSWP.29.pdf,pdf,7.0,doc_00009_chunk_001,Fig. 1. CSF Core structure The CSF Core Functi...
3,What steps are described for incident response...,1,0.774513,nist.sp.800-61r2.pdf,pdf,5.0,doc_00039_chunk_001,Keywords computer security incident; incident ...
4,What steps are described for incident response...,2,0.771700,nist.sp.800-61r2.pdf,pdf,28.0,doc_00062_chunk_002,"reporting incidents, among other items.  Deve..."
5,What steps are described for incident response...,3,0.747258,nist.sp.800-61r2.pdf,pdf,51.0,doc_00085_chunk_003,contain (5) and eradicate (6) the incident for...
6,What kind of system security planning informat...,1,0.702003,cui-ssp-template-final.docx,docx,NaN,doc_00001_chunk_000,<<Insert name>> SYSTEM SECURITY PLAN Last Upda...
7,What kind of system security planning informat...,2,0.643181,cui-ssp-template-final.docx,docx,NaN,doc_00001_chunk_014,Not Applicable Current implementation or plan...
8,What kind of system security planning informat...,3,0.635334,cui-ssp-template-final.docx,docx,NaN,doc_00001_chunk_034,Implemented Planned to be Implemented Not Ap...
9,What information is present in the audit trail...,1,0.711440,audit.txt,txt,NaN,doc_00000_chunk_028,Review of Audit Trails Audit trails can be use...


In [28]:
# ============================================================
# STEP 24 — Save outputs
# ============================================================

embedding_summary_path = OUTPUT_DIR / "embedding_summary.csv"
results_path = OUTPUT_DIR / "single_query_results.csv"
multi_query_path = OUTPUT_DIR / "multi_query_results.csv"

embedding_summary_df.to_csv(embedding_summary_path, index=False)
results_df.to_csv(results_path, index=False)
multi_query_df.to_csv(multi_query_path, index=False)

print("Saved:", embedding_summary_path)
print("Saved:", results_path)
print("Saved:", multi_query_path)

Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\embedding_summary.csv
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\single_query_results.csv
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\multi_query_results.csv


In [29]:
# ============================================================
# STEP 25 — Save chunk metadata snapshot
# ============================================================

chunk_metadata_rows = []

for record in embedded_records:
    meta = record["metadata"]
    chunk_metadata_rows.append({
        "chunk_id": meta.get("chunk_id"),
        "parent_document_id": meta.get("parent_document_id"),
        "file_name": meta.get("file_name"),
        "doc_type": meta.get("doc_type"),
        "page_number": meta.get("page_number"),
        "source_path": meta.get("source_path")
    })

chunk_metadata_df = pd.DataFrame(chunk_metadata_rows)
chunk_metadata_path = OUTPUT_DIR / "chunk_metadata_snapshot.csv"

chunk_metadata_df.to_csv(chunk_metadata_path, index=False)
print("Saved:", chunk_metadata_path)

Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\chunk_metadata_snapshot.csv


In [30]:
# ============================================================
# STEP 26 — Final conclusion
# ============================================================

print("Notebook 04 completed.")
print("Total cleaned documents:", len(cleaned_documents))
print("Total chunked documents:", len(chunked_documents))
print("Total embedded records:", len(embedded_records))
print("Embedding dimension:", embedding_dim)
print("Qdrant collection:", COLLECTION_NAME)

print("\nNext notebook: 05_retrieval_quality_checks.ipynb")

Notebook 04 completed.
Total cleaned documents: 114
Total chunked documents: 613
Total embedded records: 613
Embedding dimension: 768
Qdrant collection: cybersecurity_rag_chunks

Next notebook: 05_retrieval_quality_checks.ipynb
